# PayShield AI — ML Dataset Preparation

## AI-Powered Payment Success Optimization

### Objective

Prepare the engineered monitoring data for machine learning.

The model should learn from observable payment signals such as:

- Failure rate
- Timeout rate
- Average latency
- P95 latency
- Bank error rate
- Transaction volume
- Time-based patterns

The actual bank health condition will be used only as the target
for evaluation and will NOT be used as a model feature.

In [4]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv(
    "../data/processed/payment_monitoring_features.csv"
)

df["time_window"] = pd.to_datetime(
    df["time_window"]
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (50984, 15)


In [6]:
df = df.sort_values(
    ["receiver_bank", "time_window"]
).reset_index(drop=True)

df.head()

,receiver_bank,time_window,transaction_count,failure_rate,timeout_rate,avg_latency,max_latency,p95_latency,bank_error_rate,avg_amount,max_amount,hour,day_of_week,is_weekend,bank_health
0,AXIS,2026-01-01 00:00:00,1,0.0,0.0,1447.700,1447.70,1447.7000,0.0,345.00,345.00,0,3,0,NORMAL
1,AXIS,2026-01-01 00:55:00,2,0.0,0.0,881.935,883.95,883.7485,0.0,244.72,320.59,0,3,0,NORMAL
2,AXIS,2026-01-01 01:00:00,1,0.0,0.0,882.880,882.88,882.8800,0.0,321.00,321.00,1,3,0,NORMAL
3,AXIS,2026-01-01 01:25:00,1,0.0,0.0,751.670,751.67,751.6700,0.0,395.70,395.70,1,3,0,NORMAL
4,AXIS,2026-01-01 01:35:00,1,0.0,0.0,1161.920,1161.92,1161.9200,0.0,32.14,32.14,1,3,0,NORMAL


In [7]:
df["previous_failure_rate"] = (
    df.groupby("receiver_bank")["failure_rate"]
    .shift(1)
)

In [8]:
df["previous_latency"] = (
    df.groupby("receiver_bank")["avg_latency"]
    .shift(1)
)

In [9]:
df["previous_timeout_rate"] = (
    df.groupby("receiver_bank")["timeout_rate"]
    .shift(1)
)

In [10]:
df["failure_rate_change"] = (
    df["failure_rate"]
    - df["previous_failure_rate"]
)

In [11]:
df["latency_change"] = (
    df["avg_latency"]
    - df["previous_latency"]
)

In [12]:
df["timeout_rate_change"] = (
    df["timeout_rate"]
    - df["previous_timeout_rate"]
)

In [13]:
df["rolling_failure_rate"] = (
    df.groupby("receiver_bank")["failure_rate"]
    .transform(
        lambda x: x.rolling(
            window=3,
            min_periods=1
        ).mean()
    )
)

In [14]:
df["rolling_latency"] = (
    df.groupby("receiver_bank")["avg_latency"]
    .transform(
        lambda x: x.rolling(
            window=3,
            min_periods=1
        ).mean()
    )
)

In [15]:
df["rolling_timeout_rate"] = (
    df.groupby("receiver_bank")["timeout_rate"]
    .transform(
        lambda x: x.rolling(
            window=3,
            min_periods=1
        ).mean()
    )
)

In [16]:
df["future_health_5min"] = (
    df.groupby("receiver_bank")["bank_health"]
    .shift(-1)
)

df["future_health_10min"] = (
    df.groupby("receiver_bank")["bank_health"]
    .shift(-2)
)

df["future_health_15min"] = (
    df.groupby("receiver_bank")["bank_health"]
    .shift(-3)
)

In [17]:
future_risk = (
    df[
        [
            "future_health_5min",
            "future_health_10min",
            "future_health_15min"
        ]
    ]
    .isin(["DEGRADED", "SEVERE"])
    .any(axis=1)
)

df["risk_target"] = future_risk.astype(int)

In [18]:
print(
    df["risk_target"].value_counts()
)

risk_target
0    50702
1      282
Name: count, dtype: int64


In [19]:
print(
    df["risk_target"]
    .value_counts(normalize=True)
    * 100
)

risk_target
0    99.446885
1     0.553115
Name: proportion, dtype: float64


In [20]:
print(
    "Missing values before cleaning:"
)

print(
    df.isnull().sum()
)

Missing values before cleaning:
receiver_bank             0
time_window               0
transaction_count         0
failure_rate              0
timeout_rate              0
avg_latency               0
max_latency               0
p95_latency               0
bank_error_rate           0
avg_amount                0
max_amount                0
hour                      0
day_of_week               0
is_weekend                0
bank_health               0
previous_failure_rate    10
previous_latency         10
previous_timeout_rate    10
failure_rate_change      10
latency_change           10
timeout_rate_change      10
rolling_failure_rate      0
rolling_latency           0
rolling_timeout_rate      0
future_health_5min       10
future_health_10min      20
future_health_15min      30
risk_target               0
dtype: int64


In [21]:
df_ml = df.dropna().copy()

print(
    "ML dataset shape:",
    df_ml.shape
)

ML dataset shape: (50944, 28)


In [32]:
import pandas as pd

monitoring = pd.read_csv(
    r"C:\Users\Abhilasha\Projects\PayShield AI\data\processed\payment_monitoring_features.csv"
)

print("Monitoring loaded successfully!")
print("Shape:", monitoring.shape)

Monitoring loaded successfully!
Shape: (50984, 15)


In [33]:
# Convert time_window to datetime
monitoring["time_window"] = pd.to_datetime(
    monitoring["time_window"]
)

# Sort chronologically for each bank
monitoring = monitoring.sort_values(
    ["receiver_bank", "time_window"]
).reset_index(drop=True)

print("Sorted successfully.")
print(monitoring.head())

Sorted successfully.
  receiver_bank         time_window  transaction_count  failure_rate  \
0          AXIS 2026-01-01 00:00:00                  1           0.0   
1          AXIS 2026-01-01 00:55:00                  2           0.0   
2          AXIS 2026-01-01 01:00:00                  1           0.0   
3          AXIS 2026-01-01 01:25:00                  1           0.0   
4          AXIS 2026-01-01 01:35:00                  1           0.0   

   timeout_rate  avg_latency  max_latency  p95_latency  bank_error_rate  \
0           0.0     1447.700      1447.70    1447.7000              0.0   
1           0.0      881.935       883.95     883.7485              0.0   
2           0.0      882.880       882.88     882.8800              0.0   
3           0.0      751.670       751.67     751.6700              0.0   
4           0.0     1161.920      1161.92    1161.9200              0.0   

   avg_amount  max_amount  hour  day_of_week  is_weekend bank_health  
0      345.00      345.0

In [35]:
monitoring["previous_failure_rate"] = (
    monitoring
    .groupby("receiver_bank")["failure_rate"]
    .shift(1)
)

monitoring["previous_latency"] = (
    monitoring
    .groupby("receiver_bank")["avg_latency"]
    .shift(1)
)

monitoring["previous_timeout_rate"] = (
    monitoring
    .groupby("receiver_bank")["timeout_rate"]
    .shift(1)
)

print("Previous-window features created!")

print(
    monitoring[
        [
            "receiver_bank",
            "time_window",
            "failure_rate",
            "previous_failure_rate",
            "avg_latency",
            "previous_latency"
        ]
    ].head(15)
)

Previous-window features created!
   receiver_bank         time_window  failure_rate  previous_failure_rate  \
0           AXIS 2026-01-01 00:00:00           0.0                    NaN   
1           AXIS 2026-01-01 00:55:00           0.0                    0.0   
2           AXIS 2026-01-01 01:00:00           0.0                    0.0   
3           AXIS 2026-01-01 01:25:00           0.0                    0.0   
4           AXIS 2026-01-01 01:35:00           0.0                    0.0   
5           AXIS 2026-01-01 02:10:00           0.0                    0.0   
6           AXIS 2026-01-01 04:30:00           0.0                    0.0   
7           AXIS 2026-01-01 04:45:00           0.0                    0.0   
8           AXIS 2026-01-01 05:05:00           0.0                    0.0   
9           AXIS 2026-01-01 05:20:00           0.0                    0.0   
10          AXIS 2026-01-01 05:40:00           0.0                    0.0   
11          AXIS 2026-01-01 05:45:00      

In [37]:
monitoring["failure_rate_change"] = (
    monitoring["failure_rate"]
    - monitoring["previous_failure_rate"]
)

monitoring["latency_change"] = (
    monitoring["avg_latency"]
    - monitoring["previous_latency"]
)

monitoring["timeout_rate_change"] = (
    monitoring["timeout_rate"]
    - monitoring["previous_timeout_rate"]
)

print("Change features created!")

print(
    monitoring[
        [
            "receiver_bank",
            "time_window",
            "failure_rate_change",
            "latency_change",
            "timeout_rate_change"
        ]
    ].head(15)
)

Change features created!
   receiver_bank         time_window  failure_rate_change  latency_change  \
0           AXIS 2026-01-01 00:00:00                  NaN             NaN   
1           AXIS 2026-01-01 00:55:00                  0.0        -565.765   
2           AXIS 2026-01-01 01:00:00                  0.0           0.945   
3           AXIS 2026-01-01 01:25:00                  0.0        -131.210   
4           AXIS 2026-01-01 01:35:00                  0.0         410.250   
5           AXIS 2026-01-01 02:10:00                  0.0        -452.220   
6           AXIS 2026-01-01 04:30:00                  0.0          -0.270   
7           AXIS 2026-01-01 04:45:00                  0.0        1069.660   
8           AXIS 2026-01-01 05:05:00                  0.0        -750.270   
9           AXIS 2026-01-01 05:20:00                  0.0        -396.300   
10          AXIS 2026-01-01 05:40:00                  0.0        1651.110   
11          AXIS 2026-01-01 05:45:00               

In [38]:
monitoring["rolling_failure_rate"] = (
    monitoring
    .groupby("receiver_bank")["failure_rate"]
    .transform(
        lambda x: x.rolling(
            window=3,
            min_periods=1
        ).mean()
    )
)

monitoring["rolling_latency"] = (
    monitoring
    .groupby("receiver_bank")["avg_latency"]
    .transform(
        lambda x: x.rolling(
            window=3,
            min_periods=1
        ).mean()
    )
)

monitoring["rolling_timeout_rate"] = (
    monitoring
    .groupby("receiver_bank")["timeout_rate"]
    .transform(
        lambda x: x.rolling(
            window=3,
            min_periods=1
        ).mean()
    )
)

print("Rolling features created!")

print(
    monitoring[
        [
            "receiver_bank",
            "time_window",
            "rolling_failure_rate",
            "rolling_latency",
            "rolling_timeout_rate"
        ]
    ].head(15)
)

Rolling features created!
   receiver_bank         time_window  rolling_failure_rate  rolling_latency  \
0           AXIS 2026-01-01 00:00:00                   0.0      1447.700000   
1           AXIS 2026-01-01 00:55:00                   0.0      1164.817500   
2           AXIS 2026-01-01 01:00:00                   0.0      1070.838333   
3           AXIS 2026-01-01 01:25:00                   0.0       838.828333   
4           AXIS 2026-01-01 01:35:00                   0.0       932.156667   
5           AXIS 2026-01-01 02:10:00                   0.0       874.430000   
6           AXIS 2026-01-01 04:30:00                   0.0       860.350000   
7           AXIS 2026-01-01 04:45:00                   0.0      1066.073333   
8           AXIS 2026-01-01 05:05:00                   0.0      1172.446667   
9           AXIS 2026-01-01 05:20:00                   0.0      1146.810000   
10          AXIS 2026-01-01 05:40:00                   0.0      1314.990000   
11          AXIS 2026-01-0

In [39]:
# ==========================================
# HANDLE MISSING VALUES
# ==========================================

feature_columns = [
    "previous_failure_rate",
    "previous_latency",
    "previous_timeout_rate",
    "failure_rate_change",
    "latency_change",
    "timeout_rate_change"
]

# Fill previous/change features
monitoring[feature_columns] = (
    monitoring[feature_columns]
    .fillna(0)
)

# Rolling features
rolling_columns = [
    "rolling_failure_rate",
    "rolling_latency",
    "rolling_timeout_rate"
]

monitoring[rolling_columns] = (
    monitoring[rolling_columns]
    .fillna(0)
)

print("Missing values handled successfully.")

print(
    monitoring[
        feature_columns + rolling_columns
    ].isna().sum()
)

Missing values handled successfully.
previous_failure_rate    0
previous_latency         0
previous_timeout_rate    0
failure_rate_change      0
latency_change           0
timeout_rate_change      0
rolling_failure_rate     0
rolling_latency          0
rolling_timeout_rate     0
dtype: int64


In [40]:
print("Final monitoring shape:")
print(monitoring.shape)

print("\nFinal columns:")
print(monitoring.columns.tolist())

Final monitoring shape:
(50984, 24)

Final columns:
['receiver_bank', 'time_window', 'transaction_count', 'failure_rate', 'timeout_rate', 'avg_latency', 'max_latency', 'p95_latency', 'bank_error_rate', 'avg_amount', 'max_amount', 'hour', 'day_of_week', 'is_weekend', 'bank_health', 'previous_failure_rate', 'previous_latency', 'previous_timeout_rate', 'failure_rate_change', 'latency_change', 'timeout_rate_change', 'rolling_failure_rate', 'rolling_latency', 'rolling_timeout_rate']


In [42]:
import os

# Create processed directory if it does not exist
os.makedirs("data/processed", exist_ok=True)

# Save enhanced monitoring dataset
output_path = "data/processed/payment_monitoring_enhanced.csv"

monitoring.to_csv(
    output_path,
    index=False
)

print("Saved successfully!")
print("File:", output_path)
print("Shape:", monitoring.shape)

Saved successfully!
File: data/processed/payment_monitoring_enhanced.csv
Shape: (50984, 24)


In [43]:
import os

print(os.path.exists("data/processed/payment_monitoring_enhanced.csv"))

True


In [45]:
import pandas as pd

monitoring = pd.read_csv(
    "data/processed/payment_monitoring_enhanced.csv"
)

print("Enhanced monitoring loaded!")
print("Shape:", monitoring.shape)
print("Columns:")
print(monitoring.columns.tolist())

Enhanced monitoring loaded!
Shape: (50984, 24)
Columns:
['receiver_bank', 'time_window', 'transaction_count', 'failure_rate', 'timeout_rate', 'avg_latency', 'max_latency', 'p95_latency', 'bank_error_rate', 'avg_amount', 'max_amount', 'hour', 'day_of_week', 'is_weekend', 'bank_health', 'previous_failure_rate', 'previous_latency', 'previous_timeout_rate', 'failure_rate_change', 'latency_change', 'timeout_rate_change', 'rolling_failure_rate', 'rolling_latency', 'rolling_timeout_rate']


In [46]:
print(monitoring[
    [
        "receiver_bank",
        "time_window",
        "failure_rate",
        "timeout_rate",
        "avg_latency",
        "rolling_failure_rate",
        "rolling_latency",
        "rolling_timeout_rate",
        "bank_health"
    ]
].head())

  receiver_bank          time_window  failure_rate  timeout_rate  avg_latency  \
0          AXIS  2026-01-01 00:00:00           0.0           0.0     1447.700   
1          AXIS  2026-01-01 00:55:00           0.0           0.0      881.935   
2          AXIS  2026-01-01 01:00:00           0.0           0.0      882.880   
3          AXIS  2026-01-01 01:25:00           0.0           0.0      751.670   
4          AXIS  2026-01-01 01:35:00           0.0           0.0     1161.920   

   rolling_failure_rate  rolling_latency  rolling_timeout_rate bank_health  
0                   0.0      1447.700000                   0.0      NORMAL  
1                   0.0      1164.817500                   0.0      NORMAL  
2                   0.0      1070.838333                   0.0      NORMAL  
3                   0.0       838.828333                   0.0      NORMAL  
4                   0.0       932.156667                   0.0      NORMAL  


In [44]:
print(monitoring.columns.tolist())

['receiver_bank', 'time_window', 'transaction_count', 'failure_rate', 'timeout_rate', 'avg_latency', 'max_latency', 'p95_latency', 'bank_error_rate', 'avg_amount', 'max_amount', 'hour', 'day_of_week', 'is_weekend', 'bank_health', 'previous_failure_rate', 'previous_latency', 'previous_timeout_rate', 'failure_rate_change', 'latency_change', 'timeout_rate_change', 'rolling_failure_rate', 'rolling_latency', 'rolling_timeout_rate']


In [ ]:
X = df_ml[feature_columns]

y = df_ml["risk_target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (50944, 21)
y shape: (50944,)


In [ ]:
X.head()

,transaction_count,failure_rate,timeout_rate,avg_latency,max_latency,p95_latency,bank_error_rate,avg_amount,max_amount,hour,...,is_weekend,previous_failure_rate,previous_latency,previous_timeout_rate,failure_rate_change,latency_change,timeout_rate_change,rolling_failure_rate,rolling_latency,rolling_timeout_rate
1,2,0.0,0.0,881.935,883.95,883.7485,0.0,244.72,320.59,0,...,0,0.0,1447.700,0.0,0.0,-565.765,0.0,0.0,1164.817500,0.0
2,1,0.0,0.0,882.880,882.88,882.8800,0.0,321.00,321.00,1,...,0,0.0,881.935,0.0,0.0,0.945,0.0,0.0,1070.838333,0.0
3,1,0.0,0.0,751.670,751.67,751.6700,0.0,395.70,395.70,1,...,0,0.0,882.880,0.0,0.0,-131.210,0.0,0.0,838.828333,0.0
4,1,0.0,0.0,1161.920,1161.92,1161.9200,0.0,32.14,32.14,1,...,0,0.0,751.670,0.0,0.0,410.250,0.0,0.0,932.156667,0.0
5,1,0.0,0.0,709.700,709.70,709.7000,0.0,920.47,920.47,2,...,0,0.0,1161.920,0.0,0.0,-452.220,0.0,0.0,874.430000,0.0


In [ ]:
print("Target distribution:")
print(
    y.value_counts()
)

print("\nTarget percentage:")
print(
    y.value_counts(normalize=True) * 100
)

Target distribution:
risk_target
0    50662
1      282
Name: count, dtype: int64

Target percentage:
risk_target
0    99.446451
1     0.553549
Name: proportion, dtype: float64


In [ ]:
ml_dataset = df_ml[
    feature_columns + ["risk_target"]
].copy()

ml_dataset.to_csv(
    "../data/processed/ml_dataset.csv",
    index=False
)

print("ML dataset saved successfully!")
print("Shape:", ml_dataset.shape)

ML dataset saved successfully!
Shape: (50944, 22)
